# Robuste IP-Geolokalisierung — Ergebnis-Dashboard

Reproduzierbares Narrativ über alle Befunde (FF1–FF4). Lädt die in `eval/out/`
erzeugten Artefakte (Tabellen + Abbildungen) und stellt sie geordnet dar.

**Reihenfolge:** Daten/Quellen → E1 Genauigkeit (FF1) → T1 Bootstrap-CIs (FF1) →
Quellen-Unabhängigkeit & Default-Schwerpunkte → T6 Linien + Konfidenz-Matrix + Kalibrierung →
E2 Breakdown (FF2) → E3 Stichprobengröße (FF3) → E4 Stratifizierung → E5 Brätz-Kritik (T5) → Fazit.

> Voraussetzung: die Experiment-Skripte wurden gelaufen (`python experiments/exp_*.py`),
> sodass `eval/out/` befüllt ist.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
OUT = ROOT / "eval" / "out"
pd.set_option("display.max_rows", 30)

def show(name, width=720):
    p = OUT / name
    display(Image(filename=str(p), width=width)) if p.exists() else print("fehlt:", name)

def table(name, **kw):
    p = OUT / name
    return pd.read_csv(p, **kw) if p.exists() else print("fehlt:", name)

## Daten & Quellen
Ground Truth: RIPE-Atlas-Anchors (Hauptauswertung) sowie ein RIPE-Atlas-Probe-Stresstest.
Eingangsdaten sind **pseudonymisiert** (IP-Adressen entfernt; Koordinaten erhalten);
die Ergebnis-Tabellen/-Abbildungen liegen in `eval/out/` bzw. `eval/out_probes/`.

In [ ]:
import pandas as pd
obs = pd.read_csv(ROOT/'data'/'cache'/'observations_probes.csv')
print(f'Beispiel-Eingangsdaten (Probes, pseudonymisiert): {len(obs)} Beobachtungen, '
      f"{obs['source'].nunique()} Quellen, {obs['ip'].nunique()} Knoten.")
print('Ergebnis-Tabellen/-Abbildungen der Hauptauswertung (Anchors) liegen in eval/out/.')

## E1 — Grundgenauigkeit (FF1)
Median-/Q3-/p90-Fehler je Schätzer und je Einzelquelle. Robuste Aggregation schlägt
7 von 8 Einzelquellen; ipinfo ist (ex post) die beste Einzelquelle.

In [ ]:
display(table("e1_accuracy.csv", index_col=0).round(1))
show("e1_accuracy_cdf.png")

## T1 — Bootstrap-Konfidenzintervalle (FF1)
Paired Bootstrap (B=10000, seed=0) auf die Differenz der Mediane (robuste Aggregation − Referenz), je Einzelquelle und gegen die naive Baseline. `delta_km` < 0 = Aggregation genauer; CI ohne 0 = signifikant.

Zusätzlich gepaarte CIs auf **Mittelwert- und Tail-Raten-Differenzen** gegen die im Mittel/Tail führenden Quellen: der Nachteil gegen ipinfo.io ist signifikant, der scheinbare Mittelwert-Nachteil gegen IP2Location LITE nicht (CI enthält 0).

In [ ]:
display(table("t1_bootstrap_ci.csv").round(1))
print("\nMittelwert- und Tail-Raten-Differenzen (L1·b − Quelle), gepaart, B=10000:")
display(table("t1_bootstrap_meantail_ci.csv"))

## Quellen-Unabhängigkeit & Default-Schwerpunkte
geojs ≡ reallyfreegeoip ≡ GeoLite2 (gemessen 0 km Distanz) → eine Linie.
MaxMind `accuracy_radius` trennt empirisch **~8 km (echte Stadt) von ~558 km (Default)**.

In [ ]:
show("source_correlation.png")

## T6 — Linien-Gewichtung & Konfidenzmaß (Stütz-Konzentration)
(1) Anbieter-Linien-Kollaps (L0→L1) senkt den Tail-Fehler ~20 %. (2) Methodik-Halbierung
(L2) bringt nichts (Nullergebnis). (3) Konfidenz: die **linien-gewichtete Stütz-Konzentration S**
(Anteil entdoppelter Quellenmasse im 50-km-Kern um den Schätzer) verdreifacht out-of-fold die
Kalibrierungsgüte des früheren 2D-Labels (Streuung × Hub) — BSS +0,068 → +0,210 — und subsumiert
den Hub-Flag. Das 2D-Label (Heatmap unten) bleibt als Vergleichsanker.
FF1: Aggregation schlägt das Feld, nicht die ex-post-beste Quelle (common-mode-Untergrenze).

In [ ]:
show("t6_forest.png"); show("t6_confidence_heatmap.png", 820); show("t6_ecdf.png")
display(table("t6_point_estimators.csv").groupby(["level","variant"]).error_km
        .agg(median="median", mean="mean").round(1).head(12))

### Konfidenzmaß: Stütz-Konzentration vs. 2D-Vorgängerlabel (FF4)
Out-of-fold (10-fach, seed=0): die linien-gewichtete **Stütz-Konzentration S** erreicht
Brier Skill Score **+0,210** (vs. +0,068 für das 2D-Vorgängerlabel Streuung × Hub), ECE 0,017;
der Vorsprung ist seed-stabil und bootstrap-gesichert (Δ +0,141, 95%-CI [+0,073, +0,201]).
Als Triage-Flag dominiert S die Precision/Recall-Front: bei gleicher Flag-Quote (41 %) Recall
**92 % statt 72 %**. Gewichtet > ungewichtet bei jedem Radius → re-validiert die Linien-Gewichtung
unabhängig. Skript: `experiments/exp_support_concentration.py`.

In [ ]:
show("support_concentration.png", 920)                       # OOF-BSS-Leiter + Reliability von S
display(table("support_concentration_oof.csv"))
# 2D-Vorgängerlabel als Vergleichsanker:
show("label_calibration.png")
display(table("label_calibration_quadrants_tau100.csv").round(3))

## E2 — Kontamination / Breakdown (FF2)
Naiver Mittelwert: linear ab α>0 (Breakdown 0 %). Geom. Median: robust bis α_eff≈0,375,
bricht bei ~50 % (empirisch bestätigt). α_eff-Panel: nominale Stufen → tatsächliche Kontamination.

In [ ]:
show("e2_breakdown.png"); display(table("e2_breakdown.csv"))

## E3 — Stichprobengröße (FF3)
Robuste Schätzer schon bei n=3 brauchbar (~6 km) → kein Brätz-n≈15 nötig.
Naiver Mittelwert wird mit n **schlechter** (mehr Quellen = mehr Defaults).

In [ ]:
show("e3_samplesize.png"); display(table("e3_samplesize.csv"))

## E4 — Stratifizierung (Schwierigkeit & Region)
easy = Gleichstand (Negativ-Kontrolle), uneinig = geom. Median klar vorn, hart = Grenze.
Region: geom. Median überall am besten; ZA am härtesten, US/DE/NL exzellent.

In [ ]:
show("e4_difficulty.png", 820)
display(table("e4_difficulty.csv"))
display(table("e4_region.csv", index_col=0))

## E5 / T5 — Brätz-Schätzer, kritische Prüfung
Brätz unterliegt den robusten Schätzern. Kern-Mechanismus = **Replikat-Dominanz**
(3 identische MaxMind-Replikate nageln die Modalklasse fest, 54 %). Folge: künstlich
kleine Varianz → enges KI, das nur interne Konsistenz misst → **94,9 % Nicht-Abdeckung**
des nominalen 95-%-KI (Sicherheits-Illusion).

**Kontrollexperiment (Linien-Kollabierung):** Brätz auf 6 statt 8 Quellen (MaxMind-Familie = 1 Vertreter)
bestätigt die Replikat-Dominanz kausal — uneinig-Median 64→40 km —, während die MaxMind-Einrastquote (~54 %)
unverändert bleibt. Der Mechanismus ist real, aber unvollständig (geom. Median: ~7 km).

In [ ]:
show("e5_braetz_ci_vs_gt.png"); show("e5_braetz_qq.png")
df = table("e5_braetz.csv")
print(f"Nicht-Abdeckung 95%-KI: {100*(df.err_braetz>df.ci_km).mean():.1f}%")
print(f"Brätz <25 km am MaxMind-Wert: {100*(df.d_maxmind<25).mean():.0f}%")
print(df.groupby("bucket")[["err_braetz","err_geomed"]].median().round(1))

# Kontrollexperiment: Linien-Kollabierung (isoliert die Replikat-Dominanz)
lc = table("e5_braetz_linecollapse.csv")
print("\nKontrollexperiment — Brätz: 8 Quellen (full) vs. 6 linien-kollabiert (coll), Median km:")
display(lc.groupby("bucket")[["err_full","err_coll"]].median().round(1))
print(f"MaxMind-Einrast <25 km: full {100*(lc.mm_full<25).mean():.1f}%  ->  coll {100*(lc.mm_coll<25).mean():.1f}%")

## Reproduzierbarkeit — Determinismus-Test
Die Schätzer sind deterministisch und alle stochastischen Schritte (Bootstrap, Kontamination, out-of-fold-Folds) nutzen feste Seeds (Methodik 3.4). Der Test belegt das maschinell.

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest", "tests/test_determinism.py", "-q"],
                   cwd=ROOT, capture_output=True, text=True)
print(r.stdout[-1500:] or r.stderr[-1500:])

## Probe-Vergleich — RIPE-Atlas-Probes
Separater, explorativer Stresstest gegen residentielle/NAT/mobile Anschlüsse (**direktionaler Stresstest**; Probe-GT selbstgemeldet/gerundet). Anchor-Spalte reproduziert Kapitel 4; Probe-Aggregation L1·b: Median 5,4 / Mittel 101,9 km. Tag-Gradient: datacentre 4,6 < home/nat 8,5 < mobile 41,2 km.

In [ ]:
OUTP = ROOT / "eval" / "out_probes"
def ptab(name):
    q = OUTP / name
    return pd.read_csv(q) if q.exists() else print("fehlt:", name)
print("Anchor vs Probe je Schätzer/Quelle (Median/Mittel/Tail; Anchor-Hauptauswertung):")
display(ptab("probes_vs_anchors.csv"))
print("Probe-Stratifizierung nach Anschlussklasse (Aggregation L1·b):")
display(ptab("probes_by_tag.csv"))

## Fazit (FF1–FF4)
- **FF1:** robuste Aggregation ist klar genauer als der naive Mittelwert und schlägt fast alle Einzelquellen.
- **FF2:** geom. Median hält bis ~50 % Kontamination, der Mittelwert bricht ab α>0.
- **FF3:** robuste Schätzer tragen schon bei n=3; mehr (korrelierte/defaulte) Quellen schaden dem Mittel.
- **FF4:** Provenance-Kette + das linien-gewichtete **Konfidenzmaß S** (Stütz-Konzentration) liefern eine *qualifizierte* Schätzung — S ist out-of-fold kalibriert (BSS +0,210, ECE 0,017) und verdreifacht die Güte des 2D-Vorgängerlabels (Streuung × Hub); Brätz' KI dagegen ist systematisch fehlkalibriert (94,9 % Nicht-Abdeckung).
- **Beitrag:** ein Linien-Gewichtungs-Prinzip, das dieselbe Entdopplung sowohl für die Punktschätzung (gegen Replikat-Dominanz) als auch für die Konfidenz (Stütz-Konzentration gegen common-mode failure) nutzt.